## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | DenseNet-121 original published crops training with plain CrossEntropyLoss |
| Model / workflow | DenseNet-121 |
| Input | Original published 224x224 crops (224x224 → resized to 384) |
| Loss | Plain CrossEntropyLoss |
| Training / pipeline | 3-stage (frozen → coarse-tune → fine-tune), CE loss, CosineAnnealingLR |
| Improvements | Plain DataLoader matching the working baseline (cv2.imread per __getitem__, num_workers=2, default collate). No RAM preload, no persistent_workers, no channels_last. |
| Result | See the executed cells below for metrics, plots, and checkpoint details. |
| Status | training notebook (was failing mid Stage 1; root cause was RAM preload + 4 workers on a 2-worker VM — now mirrors the stable baseline) |


## Detailed config

### Identity

| Item | Value |
| --- | --- |
| Purpose | DenseNet-121 original published crops training with plain CrossEntropyLoss |
| Workflow | 3-stage (frozen → coarse-tune → fine-tune) |
| Base checkpoint | Not used (trained from scratch via ImageNet pretrained timm weights) |
| Output directory | `/content/drive/MyDrive/Models/densenet121_optimized_original<TIMESTAMP>/` |

### Dataset

| Item | Value |
| --- | --- |
| Classes | 5 KL grades (0–4) |
| Class labels | 0: Healthy, 1: Doubtful, 2: Minimal, 3: Moderate, 4: Severe |
| Dataset root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224` |
| Input size | 384×384 |
| Input source | Original published 224×224 crops → SquarePad → Resize(384) |

### Training

| Item | Value |
| --- | --- |
| Seed | 42 |
| Optimizer | AdamW with stage-specific LRs (discriminative LR for head vs backbone) |
| Weight decay | 1e-4 |
| Batch size | 48 |
| Num workers | 2 (matches the Colab VM worker limit; no persistent_workers) |
| Scheduler | CosineAnnealingLR per stage (mirrors the baseline notebook) |
| Loss | Plain CrossEntropyLoss |
| Sampler | WeightedRandomSampler (`weights = 1.0 / class_counts^SAMPLER_POWER`, `SAMPLER_POWER = 1.0`) |
| Augmentation | OpenCV CLAHE → SquarePad → PIL → HFlip(p=0.5) → Rotation(5°) → ColorJitter(b=0.08, c=0.08) → Resize(384) → RandomErasing(p=0.10) → ImageNet normalization |

**Stage schedule**

| Stage | Epochs | Head LR | Backbone LR | Scope |
| --- | --- | --- | --- | --- |
| Stage 1 | 5 | 0.0003 | — | head-only, backbone frozen |
| Stage 2 | 15 | 0.0003 | 3e-05 | head + backbone, backbone 10× lower LR |
| Stage 3 | 10 | 1e-05 | 1e-05 | full fine-tune, low LR |
| **Total** | **30** | — | — | — |

### Selection & metrics

| Item | Value |
| --- | --- |
| Selection score | QWK only |
| Metrics recorded | QWK, MAE, off-by-1 accuracy, per-class F1, macro-F1, macro-AP, macro-AUC |
| Outputs per run | best_model.pth (max selection), last_model.pth (every epoch), history.csv, run_config.json |

### Differences from the baseline notebook (2026-08-04_02_train_densenet121_original_384.ipynb)

- **Loss**: Plain CrossEntropyLoss (baseline already does this).
- **Class imbalance**: WeightedRandomSampler balances class frequencies (same as baseline).
- **Scheduler**: CosineAnnealingLR per stage (matches baseline; OneCycleLR was removed because its per-batch `scheduler.step()` was brittle on resume).
- **Evaluator**: Full metrics — QWK, MAE, off-by-1, macro-F1, macro-AP, macro-AUC, per-class F1.
- **No RAM preload, no persistent_workers, no prefetch_factor, no channels_last** — these were the root cause of the original kernel-cancel failures (RAM preload + 4 workers on a 2-worker VM). The data pipeline now matches the stable baseline: cv2.imread from Drive inside `__getitem__`, plain `DataLoader(num_workers=2)`, default collate.



# DenseNet-121 Original Published Crops — Plain CrossEntropy Training

Built on `02_train_densenet121_original_384.ipynb` (CE baseline). Key changes:

- **Class imbalance**: WeightedRandomSampler balances class frequencies
- **Scheduler**: CosineAnnealingLR per stage (matches the baseline; OneCycleLR was removed because its per-batch `scheduler.step()` was brittle)
- **DataLoader**: Plain `num_workers=2` with default collate — mirrors the working baseline. No RAM preload, no persistent_workers, no channels_last.
- **Evaluator**: Full metrics (QWK used for selection only)

Dataset: `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224`
Input: original published 224x224 crops → SquarePad → Resize(384)


## 0. Setup
Install dependencies and mount drive.

In [1]:
!pip -q install "timm>=1.0" "h5py>=3.9"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
import copy
import os
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score, cohen_kappa_score,
    precision_recall_fscore_support, roc_auc_score,
    mean_absolute_error, classification_report
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Mounted at /content/drive


## 1. Configuration

Mirrors the CE baseline config. Optimizations injected in later cells.

In [3]:
# ─── Paths ───────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224")

# ─── A100-aware training config ───────────────────────────────────────────
# Auto-detect GPU: A100 → batch=128/workers=8, otherwise batch=48/workers=2.
SEED = 42
INPUT_SIZE = 224        # native 224x224 resolution
IS_A100 = torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
BATCH_SIZE = 64 if IS_A100 else 48
NUM_WORKERS = 8 if IS_A100 else 2
PERSISTENT_WORKERS = False   # never keep workers alive — releases GPU RAM between batches
EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 15
EPOCHS_STAGE3 = 10
TOTAL_EPOCHS = EPOCHS_STAGE1 + EPOCHS_STAGE2 + EPOCHS_STAGE3

# ─── Optimizer ─────────────────────────────────────────────────────────────
WEIGHT_DECAY = 1e-4
LR_HEAD_STAGE1 = 3e-4
LR_HEAD_STAGE2 = 3e-4
LR_BACKBONE_STAGE2 = 3e-5
LR_STAGE3 = 1e-5
SAMPLER_POWER = 1.0   # WeightedRandomSampler: weights = (1.0 / class_counts^SAMPLER_POWER)

# ─── Selection ──────────────────────────────────────────────────────────────
# QWK only — proven, simple, matches ordinal-implied ordering
SEL_QWK_W = 1.0

# ─── Derived ────────────────────────────────────────────────────────────────
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = Path("/content/drive/MyDrive/Models/densenet121_optimized_original") / RUN_TIMESTAMP

for p in (DATASET_ROOT,):
    if not p.exists():
        raise FileNotFoundError(p)
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if IS_A100:
    print("A100 detected: batch=128, workers=8, persistent=True")
else:
    print(f"Batch={BATCH_SIZE}, workers={NUM_WORKERS}")
print("=" * 65)
print(" CONFIG")
print("=" * 65)
print(f"  Loss          : CrossEntropyLoss (plain, single)")
print(f"  Selection     : QWK only")
print(f"  Sampler power : {SAMPLER_POWER}")
print(f"  Scheduler     : CosineAnnealingLR per stage")
print("=" * 65)


Device: cuda
Batch=48, workers=2
 CONFIG
  Loss          : CrossEntropyLoss (plain, single)
  Selection     : QWK only
  Sampler power : 1.0
  Scheduler     : CosineAnnealingLR per stage


## 2. Load train/val splits

Load from the original published crop dataset (kneeKL224). No test split used for training.

In [4]:
# Load every train/val PNG once; the published crop dataset has zero duplicates,
# so we skip the MD5 step that the baseline used (it cost ~20 minutes on Drive).

def load_split(root, split):
    """Load one train/val split from the original published crop structure."""
    paths, labels = [], []
    for grade in range(5):
        grade_dir = root / split / str(grade)
        if not grade_dir.exists():
            continue
        for img_file in sorted(grade_dir.glob("*.png")):
            paths.append(str(img_file))
            labels.append(grade)
    print(f"  Loaded {split}: {len(paths)} images")
    return paths, labels


print(f"Dataset: {DATASET_ROOT}")
train_paths, train_labels = load_split(DATASET_ROOT, "train")
val_paths, val_labels = load_split(DATASET_ROOT, "val")

  # Class counts (used for WeightedRandomSampler)
class_counts = np.bincount(train_labels, minlength=5)
print(f"\nClass counts: {dict(enumerate(class_counts))}")


Dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224
  Loaded train: 5778 images
  Loaded val: 826 images

Class counts: {0: np.int64(2286), 1: np.int64(1046), 2: np.int64(1516), 3: np.int64(757), 4: np.int64(173)}


## 3. Preprocessing & Dataset

Same preprocessing as CE baseline: CLAHE → SquarePad → Resize(384) → normalize.

In [5]:
class OpenCVCLAHE:
    '''CLAHE on the L channel of LAB - same as the baseline notebook.'''

    def __init__(self, clip_limit=1.25, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, image_rgb):
        img_lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        lightness, channel_a, channel_b = cv2.split(img_lab)
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size
        )
        lightness = clahe.apply(lightness)
        return cv2.cvtColor(
            cv2.merge((lightness, channel_a, channel_b)),
            cv2.COLOR_LAB2RGB,
        )


class SquarePad:
    '''Pad the image to a square (no resize) - same as the baseline notebook.'''

    def __call__(self, image_rgb):
        height, width = image_rgb.shape[:2]
        side = max(height, width)
        top = (side - height) // 2
        bottom = side - height - top
        left = (side - width) // 2
        right = side - width - left
        return cv2.copyMakeBorder(
            image_rgb,
            top,
            bottom,
            left,
            right,
            borderType=cv2.BORDER_CONSTANT,
            value=[0, 0, 0],
        )


# Per-item Compose pipeline (CLAHE → SquarePad → PIL → augment → Resize → Tensor → Normalize).
# No RAM preload, no uint8 cache, no custom collate_fn — mirrors the working baseline notebook.
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)

train_transform = transforms.Compose([
    OpenCVCLAHE(clip_limit=1.25, tile_grid_size=(8, 8)),
    SquarePad(),
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    normalize,
])

val_transform = transforms.Compose([
    OpenCVCLAHE(clip_limit=1.25, tile_grid_size=(8, 8)),
    SquarePad(),
    transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    normalize,
])



In [6]:
class PublishedCropDataset(Dataset):
    """Load one split from the original published 224x224 crop dataset.

    Mirrors the baseline notebook's KaggleKneeOsteoarthritisDataset:
    - cv2.imread from Drive inside __getitem__ (no preload, no RAM cache).
    - The whole Compose pipeline (CLAHE + SquarePad + augment + Resize + normalize)
      runs per-item; workers parallelize the cv2 + PIL cost.
    """

    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image_bgr = cv2.imread(self.paths[index])
        if image_bgr is None:
            raise IOError(f"Cannot read: {self.paths[index]}")
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            image = self.transform(image)
        return image, int(self.labels[index])


# ─── Samplers & loaders ───────────────────────────────────────────────────
weights = (1.0 / np.power(class_counts, SAMPLER_POWER))[train_labels]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)

train_dataset = PublishedCropDataset(train_paths, train_labels, train_transform)
val_dataset = PublishedCropDataset(val_paths, val_labels, val_transform)

# Validation batch can be larger since there's no augmentation and no gradient memory.
VAL_BATCH_SIZE = BATCH_SIZE   # match train batch

# Same DataLoader pattern as the baseline notebook — no persistent_workers,
# no prefetch_factor, default collate_fn. This is what kept the baseline stable.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS,)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS,)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")



Train batches: 121 | Val batches: 18


## 4. Loss Function

Single component — plain CrossEntropyLoss.


## 6. Model

DenseNet-121 with standard linear head. Three freezing stages.

In [7]:
class DenseNet121Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "densenet121", pretrained=True, num_classes=5, drop_rate=0.20
        )

    @property
    def gradcam_target_layer(self):
        return self.backbone.features.norm5

    def forward(self, images):
        return self.backbone(images)

    def freeze_all(self):
        for p in self.parameters():
            p.requires_grad = False
        for p in self.backbone.classifier.parameters():
            p.requires_grad = True
        print("  [Stage 1] Frozen backbone, training classifier head only")

    def unfreeze_last_block(self):
        for p in self.parameters():
            p.requires_grad = False
        for name, m in self.backbone.named_modules():
            if any(x in name for x in ["denseblock3", "denseblock4", "norm5"]):
                for p in m.parameters():
                    p.requires_grad = True
        for p in self.backbone.classifier.parameters():
            p.requires_grad = True
        print("  [Stage 2] Unfrozen last dense block(s), training last block + classifier")

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True
        print("  [Stage 3] Full model unfrozen, fine-tuning end-to-end")


model = DenseNet121Model().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters : {total_params:,}")
print(f"Trainable        : {trainable_params:,}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Total parameters : 6,958,981
Trainable        : 6,958,981


## 7. Evaluation Helper

Full metrics displayed inline — QWK, MAE, off-by-1 accuracy, macro-F1, macro-AP, macro-AUC, and per-class F1.


In [8]:
def evaluate_full(loader):
    """Run the model on `loader` and return all metrics.

    Displays: QWK, MAE, off-by-1 accuracy, macro-F1, macro-AP, macro-AUC, per-class F1.
    """
    model.eval()
    all_labels, all_preds, all_probas = [], [], []
    total_loss, total_samples = 0.0, 0

    with torch.inference_mode():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images).float()
            loss = F.cross_entropy(logits, labels)
            probas = F.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probas.extend(probas)
            total_loss += loss.item() * len(labels)
            total_samples += len(labels)

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_preds)
    y_proba = np.asarray(all_probas)
    y_onehot = np.eye(5)[y_true]

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    off1_acc = np.mean(np.abs(y_true - y_pred) <= 1)
    macro_f1, _, _, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    macro_ap = average_precision_score(y_onehot, y_proba, average="macro")
    macro_auc = roc_auc_score(y_onehot, y_proba, average="macro")
    accuracy = np.mean(y_true == y_pred)

    per_class_f1 = {}
    for grade in range(5):
        mask_t = y_true == grade
        mask_p = y_pred == grade
        tp = np.sum(mask_t & mask_p)
        fp = np.sum((y_true != grade) & mask_p)
        fn = np.sum(mask_t & (y_pred != grade))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class_f1[grade] = {"precision": precision, "recall": recall, "f1": f1, "support": int(mask_t.sum())}

    selection = qwk  # QWK-only selection

    print("\n" + "─" * 60)
    print(f"  Val Loss      : {total_loss / total_samples:.4f}")
    print(f"  Accuracy      : {accuracy:.4f}")
    print(f"  QWK           : {qwk:.4f}")
    print(f"  MAE           : {mae:.4f}")
    print(f"  Off-by-1 Acc  : {off1_acc:.4f}")
    print(f"  Macro F1      : {macro_f1:.4f}")
    print(f"  Macro AP      : {macro_ap:.4f}")
    print(f"  Macro AUC     : {macro_auc:.4f}")
    print(f"  Selection     : {selection:.4f}")
    print("\n  Per-class F1 (precision / recall / f1 / support):")
    for g, m in per_class_f1.items():
        print(f"    Grade {g}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} (n={m['support']})")
    print("─" * 60)
    print(classification_report(y_true, y_pred, target_names=[str(g) for g in range(5)], zero_division=0))
    print("─" * 60)

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy,
        "qwk": float(qwk),
        "mae": float(mae),
        "off1_acc": float(off1_acc),
        "macro_f1": float(macro_f1),
        "macro_ap": float(macro_ap),
        "macro_auc": float(macro_auc),
        "selection": float(selection),
        "per_class_f1": per_class_f1,
        "probas": y_proba,
    }


## 8. Training Loop

WeightedRandomSampler balances class frequencies. CosineAnnealingLR per stage.


In [9]:
print("\n" + "=" * 65)
print(" TRAINING CONFIG")
print("=" * 65)
print(f"  Loss               : CrossEntropyLoss")
print(f"  Scheduler          : CosineAnnealingLR (per-stage, safe across resumes)")
print(f"  Sampler            : WeightedRandomSampler (power={SAMPLER_POWER})")
print(f"  Stages             : {EPOCHS_STAGE1}/{EPOCHS_STAGE2}/{EPOCHS_STAGE3} epochs")
print("=" * 65)

history = []
best_selection = -float("inf")
best_checkpoint_path = RUN_DIR / "best_model.pth"
last_checkpoint_path = RUN_DIR / "last_model.pth"

def make_optimizer(lr, backbone_lr=None):
    if backbone_lr is None:
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    head_params = list(model.backbone.classifier.parameters())
    bb_params = [p for n, p in model.named_parameters() if "classifier" not in n and p.requires_grad]
    return torch.optim.AdamW([
        {"params": head_params, "lr": lr},
        {"params": bb_params, "lr": backbone_lr},
    ], weight_decay=WEIGHT_DECAY)


def make_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )


scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

for stage_idx, (stage_name, stage_epochs, head_lr, bb_lr) in enumerate([
    ("Stage 1 — Head Only", EPOCHS_STAGE1, LR_HEAD_STAGE1, None),
    ("Stage 2 — Coarse Tune", EPOCHS_STAGE2, LR_HEAD_STAGE2, LR_BACKBONE_STAGE2),
    ("Stage 3 — Fine-Tune", EPOCHS_STAGE3, LR_STAGE3, LR_STAGE3),
], start=1):
    print(f"\n{'=' * 65}")
    print(f" {stage_name} ({stage_epochs} epochs)")
    print(f"{'=' * 65}")

    if stage_idx == 1:
        model.freeze_all()
    elif stage_idx == 2:
        model.unfreeze_last_block()
    else:
        model.unfreeze_all()

    optimizer = make_optimizer(head_lr, bb_lr)
    scheduler = make_scheduler(optimizer, stage_epochs)

    for epoch in range(stage_epochs):
        model.train()
        running_loss, total_samples = 0.0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{stage_epochs} [Train]")
        for images, labels in pbar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = F.cross_entropy(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * len(labels)
            total_samples += len(labels)
            pbar.set_postfix(lr=f"{optimizer.param_groups[0]['lr']:.2e}", loss=f"{loss.item():.4f}")

        scheduler.step()

        metrics = evaluate_full(val_loader)
        train_loss = running_loss / total_samples

        row = {
            "stage": stage_idx,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": metrics["loss"],
            "accuracy": metrics["accuracy"],
            "qwk": metrics["qwk"],
            "mae": metrics["mae"],
            "off1_acc": metrics["off1_acc"],
            "macro_f1": metrics["macro_f1"],
            "macro_ap": metrics["macro_ap"],
            "macro_auc": metrics["macro_auc"],
            "selection": metrics["selection"],
            "lr_head": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(json.dumps({
            "stage": stage_idx, "epoch": epoch + 1,
            "train_loss": f"{train_loss:.4f}",
            "val_loss": f"{metrics['loss']:.4f}",
            "accuracy": f"{metrics['accuracy']:.4f}",
            "qwk": f"{metrics['qwk']:.4f}",
            "mae": f"{metrics['mae']:.4f}",
            "off1_acc": f"{metrics['off1_acc']:.4f}",
            "macro_f1": f"{metrics['macro_f1']:.4f}",
            "macro_ap": f"{metrics['macro_ap']:.4f}",
            "macro_auc": f"{metrics['macro_auc']:.4f}",
            "selection": f"{metrics['selection']:.4f}",
        }, indent=2))

        payload = {
            "model_state_dict": model.state_dict(),
            "architecture": "densenet121_option_a",
            "loss_type": "cross_entropy",
            "stage": stage_idx,
            "epoch": epoch + 1,
            "selection": metrics["selection"],
            "qwk": metrics["qwk"],
            "macro_f1": metrics["macro_f1"],
            "macro_ap": metrics["macro_ap"],
            "macro_auc": metrics["macro_auc"],
            "validation_metrics": metrics,
            "history": history,
            "run_timestamp": RUN_TIMESTAMP,
            "fixed_production_config": {
                "input_resize": INPUT_SIZE,
                "input_crop": INPUT_SIZE,
                "batch_size": BATCH_SIZE,
                "sampler": "weighted_inverse_frequency",
                "sampler_power": SAMPLER_POWER,
                "horizontal_flip_probability": 0.50,
                "rotation_degrees": 5,
                "color_jitter_brightness": 0.08,
                "color_jitter_contrast": 0.08,
                "random_erasing_probability": 0.10,
                "stage_epochs": [EPOCHS_STAGE1, EPOCHS_STAGE2, EPOCHS_STAGE3],
                "learning_rates": [LR_HEAD_STAGE1, LR_HEAD_STAGE2, LR_STAGE3],
                "scheduler": "cosine_annealing",
                "cam_method": "post_hoc_gradcam",
                                    },
        }

        torch.save(payload, last_checkpoint_path)

        if metrics["selection"] > best_selection:
            best_selection = metrics["selection"]
            torch.save(payload, best_checkpoint_path)
            print(f"  -> New best! Selection={best_selection:.4f} QWK={metrics['qwk']:.4f}")



 TRAINING CONFIG
  Loss               : CrossEntropyLoss
  Scheduler          : CosineAnnealingLR (per-stage, safe across resumes)
  Sampler            : WeightedRandomSampler (power=1.0)
  Stages             : 5/15/10 epochs

 Stage 1 — Head Only (5 epochs)
  [Stage 1] Frozen backbone, training classifier head only


Epoch 1/5 [Train]:   0%|          | 0/121 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
  Val Loss      : 1.4363
  Accuracy      : 0.3765
  QWK           : 0.3519
  MAE           : 0.9806
  Off-by-1 Acc  : 0.7385
  Macro F1      : 0.3271
  Macro AP      : 0.3264
  Macro AUC     : 0.6653
  Selection     : 0.3519

  Per-class F1 (precision / recall / f1 / support):
    Grade 0: P=0.503 R=0.573 F1=0.536 (n=328)
    Grade 1: P=0.205 R=0.281 F1=0.237 (n=153)
    Grade 2: P=0.420 R=0.222 F1=0.290 (n=212)
    Grade 3: P=0.253 R=0.198 F1=0.222 (n=106)
    Grade 4: P=0.255 R=0.444 F1=0.324 (n=27)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

           0       0.50      0.57      0.54       328
           1       0.20      0.28      0.24       153
           2       0.42      0.22      0.29       212
           3       0.25      0.20      0.22       106
           4       0.26      0.44      0.32        27

    accuracy                           0.38 

Epoch 2/5 [Train]:   0%|          | 0/121 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
  Val Loss      : 0.8808
  Accuracy      : 0.6114
  QWK           : 0.7725
  MAE           : 0.4697
  Off-by-1 Acc  : 0.9249
  Macro F1      : 0.6277
  Macro AP      : 0.7067
  Macro AUC     : 0.8748
  Selection     : 0.7725

  Per-class F1 (precision / recall / f1 / support):
    Grade 0: P=0.714 R=0.662 F1=0.687 (n=328)
    Grade 1: P=0.305 R=0.379 F1=0.338 (n=153)
    Grade 2: P=0.656 R=0.604 F1=0.629 (n=212)
    Grade 3: P=0.757 R=0.736 F1=0.746 (n=106)
    Grade 4: P=0.706 R=0.889 F1=0.787 (n=27)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

           0       0.71      0.66      0.69       328
           1       0.31      0.38      0.34       153
           2       0.66      0.60      0.63       212
           3       0.76      0.74      0.75       106
           4       0.71      0.89      0.79        27

    accuracy                           0.61 

Epoch 10/10 [Train]:   0%|          | 0/121 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
  Val Loss      : 0.8887
  Accuracy      : 0.6041
  QWK           : 0.7708
  MAE           : 0.4746
  Off-by-1 Acc  : 0.9274
  Macro F1      : 0.6320
  Macro AP      : 0.7077
  Macro AUC     : 0.8750
  Selection     : 0.7708

  Per-class F1 (precision / recall / f1 / support):
    Grade 0: P=0.731 R=0.622 F1=0.672 (n=328)
    Grade 1: P=0.305 R=0.399 F1=0.346 (n=153)
    Grade 2: P=0.637 R=0.604 F1=0.620 (n=212)
    Grade 3: P=0.713 R=0.774 F1=0.742 (n=106)
    Grade 4: P=0.774 R=0.889 F1=0.828 (n=27)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

           0       0.73      0.62      0.67       328
           1       0.30      0.40      0.35       153
           2       0.64      0.60      0.62       212
           3       0.71      0.77      0.74       106
           4       0.77      0.89      0.83        27

    accuracy                           0.60 

In [10]:
pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

with open(RUN_DIR / "run_config.json", "w") as f:
    json.dump({
        "architecture": "densenet121_ce_baseline",
        "loss_type": "ce_baseline",
        "input_size": INPUT_SIZE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "stage_epochs": [EPOCHS_STAGE1, EPOCHS_STAGE2, EPOCHS_STAGE3],
        "learning_rates": [LR_HEAD_STAGE1, LR_HEAD_STAGE2, LR_STAGE3],
        "best_selection": best_selection,
        "run_timestamp": RUN_TIMESTAMP,
        "history": history,
    }, f, indent=2)

print(f"\nHistory saved to {RUN_DIR / 'history.csv'}")
print(f"Run config saved to {RUN_DIR / 'run_config.json'}")
print(f"Best model:   {best_checkpoint_path}")
print(f"Last model:   {last_checkpoint_path}")
print(f"Final best selection score: {best_selection:.4f}")




History saved to /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/history.csv
Run config saved to /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/run_config.json
Best model:   /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Last model:   /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/last_model.pth
Final best selection score: 0.7725
